# Player News Ingestion - Sleeper API (Automated)

**Purpose:** Automatically fetch player news, injury status, and trending data from Sleeper API

**API:** Sleeper (Free, no authentication required)

**Endpoints:**
* Players: `https://api.sleeper.app/v1/players/nfl`
* Trending: `https://api.sleeper.app/v1/players/nfl/trending/add`

**Output Tables (all in main.fantasai):**
* `bronze_player_news_raw` - Raw API response
* `silver_player_news` - Cleaned, enriched news
* `silver_injury_reports` - Current injury status
* `silver_trending_players` - Weekly trending adds

**Schedule:** Run weekly (Tuesday after MNF)

**Data Quality:**
* News freshness check
* Injury status validation
* Duplicate detection
* Player name normalization

In [0]:
%pip install requests --quiet

print("✅ Dependencies installed")

In [0]:
import requests
import json
from datetime import datetime
import pandas as pd

print("="*80)
print("Fetching Player News from Sleeper API")
print(f"Timestamp: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print("="*80)

# Fetch all NFL players from Sleeper
url = "https://api.sleeper.app/v1/players/nfl"
print(f"\n📡 Calling: {url}")

try:
    response = requests.get(url, timeout=30)
    response.raise_for_status()
    players_data = response.json()
    
    print(f"✅ Fetched {len(players_data)} players")
    print(f"   Status code: {response.status_code}")
    print(f"   Response size: {len(response.content)} bytes")
except Exception as e:
    print(f"❌ Error fetching data: {e}")
    players_data = {}

# Filter to relevant positions and active players
relevant_positions = ['QB', 'RB', 'WR', 'TE', 'K', 'DEF']
filtered_players = []

for player_id, player in players_data.items():
    if player.get('position') in relevant_positions:
        # Extract key fields
        player_info = {
            'player_id': player_id,
            'player_name': player.get('full_name', player.get('first_name', '') + ' ' + player.get('last_name', '')).strip(),
            'first_name': player.get('first_name'),
            'last_name': player.get('last_name'),
            'position': player.get('position'),
            'team': player.get('team'),
            'status': player.get('status'),
            'injury_status': player.get('injury_status'),
            'injury_body_part': player.get('injury_body_part'),
            'injury_notes': player.get('injury_notes'),
            'injury_start_date': player.get('injury_start_date'),
            'years_exp': player.get('years_exp'),
            'active': player.get('active', True),
            'age': player.get('age'),
            'number': player.get('number'),
            'depth_chart_order': player.get('depth_chart_order'),
            'depth_chart_position': player.get('depth_chart_position'),
            'news_updated': player.get('news_updated'),
            'fantasy_positions': ','.join(player.get('fantasy_positions', [])) if player.get('fantasy_positions') else None,
            'fetched_at': datetime.now(),
            'raw_data': json.dumps(player)  # Keep full payload for reference
        }
        filtered_players.append(player_info)

df_players = pd.DataFrame(filtered_players)

print(f"\n✅ Filtered to {len(df_players)} relevant players")
print(f"\nPosition breakdown:")
if not df_players.empty:
    print(df_players['position'].value_counts())
    print(f"\n📊 Players with injuries: {df_players['injury_status'].notna().sum()}")
    print(f"📊 Players with news updates: {df_players['news_updated'].notna().sum()}")

In [0]:
# Fetch trending players (most added in past 24h)
trending_url = "https://api.sleeper.app/v1/players/nfl/trending/add?lookback_hours=24&limit=100"
print(f"\n📡 Calling trending API: {trending_url}")

try:
    trending_response = requests.get(trending_url, timeout=30)
    trending_response.raise_for_status()
    trending_data = trending_response.json()
    
    print(f"✅ Fetched {len(trending_data)} trending players")
    
    # Parse trending data
    trending_players = []
    for item in trending_data:
        trending_players.append({
            'player_id': item.get('player_id'),
            'count': item.get('count'),  # Number of adds
            'fetched_at': datetime.now()
        })
    
    df_trending = pd.DataFrame(trending_players)
    
    # Join with player info
    if not df_trending.empty and not df_players.empty:
        df_trending = df_trending.merge(
            df_players[['player_id', 'player_name', 'position', 'team']],
            on='player_id',
            how='left'
        )
        print(f"\n📈 Top 10 trending adds:")
        display(df_trending.head(10))
except Exception as e:
    print(f"⚠️ Error fetching trending data: {e}")
    df_trending = pd.DataFrame()

In [0]:
# Create injury report table from players with active injuries
if not df_players.empty:
    df_injuries = df_players[
        df_players['injury_status'].notna()
    ][[
        'player_id', 'player_name', 'position', 'team',
        'injury_status', 'injury_body_part', 'injury_notes',
        'injury_start_date', 'fetched_at'
    ]].copy()
    
    print(f"\n🏥 Extracted {len(df_injuries)} injury reports")
    if len(df_injuries) > 0:
        print(f"\nInjury status breakdown:")
        print(df_injuries['injury_status'].value_counts())
        print(f"\nMost common injuries:")
        print(df_injuries['injury_body_part'].value_counts().head(10))
else:
    df_injuries = pd.DataFrame()

In [0]:
# Create player news table from players with recent news updates
if not df_players.empty:
    df_news = df_players[
        df_players['news_updated'].notna()
    ][[
        'player_id', 'player_name', 'position', 'team',
        'news_updated', 'injury_status', 'injury_notes',
        'status', 'depth_chart_order', 'depth_chart_position',
        'fetched_at'
    ]].copy()
    
    # Convert news_updated to datetime (it's a Unix timestamp in milliseconds)
    df_news['news_updated'] = pd.to_datetime(df_news['news_updated'], unit='ms', errors='coerce')
    
    # Filter to news from past 7 days
    seven_days_ago = pd.Timestamp.now() - pd.Timedelta(days=7)
    df_news = df_news[df_news['news_updated'] >= seven_days_ago]
    
    print(f"\n📰 Extracted {len(df_news)} players with recent news (past 7 days)")
    if len(df_news) > 0:
        print(f"\nNews by position:")
        print(df_news['position'].value_counts())
else:
    df_news = pd.DataFrame()

In [0]:
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, TimestampType, BooleanType, LongType

print("\n" + "="*80)
print("Saving to Delta Tables (main.fantasai)")
print("="*80)

# 1. Save raw player data to bronze layer
if not df_players.empty:
    print("\n💾 Saving to main.fantasai.bronze_player_news_raw...")
    spark_df_raw = spark.createDataFrame(df_players)
    spark_df_raw.write.mode('overwrite').option('overwriteSchema', 'true').saveAsTable('main.fantasai.bronze_player_news_raw')
    print(f"✅ Saved {len(df_players)} players")

# 2. Save cleaned news to silver layer
if not df_news.empty:
    print("\n💾 Saving to main.fantasai.silver_player_news...")
    spark_df_news = spark.createDataFrame(df_news)
    spark_df_news.write.mode('overwrite').option('overwriteSchema', 'true').saveAsTable('main.fantasai.silver_player_news')
    print(f"✅ Saved {len(df_news)} news items")
else:
    print("\n⚠️ No recent news to save")

# 3. Save injury reports to silver layer
if not df_injuries.empty:
    print("\n💾 Saving to main.fantasai.silver_injury_reports...")
    spark_df_injuries = spark.createDataFrame(df_injuries)
    spark_df_injuries.write.mode('overwrite').option('overwriteSchema', 'true').saveAsTable('main.fantasai.silver_injury_reports')
    print(f"✅ Saved {len(df_injuries)} injury reports")
else:
    print("\n⚠️ No injuries to save")

# 4. Save trending players to silver layer
if not df_trending.empty:
    print("\n💾 Saving to main.fantasai.silver_trending_players...")
    spark_df_trending = spark.createDataFrame(df_trending)
    spark_df_trending.write.mode('overwrite').option('overwriteSchema', 'true').saveAsTable('main.fantasai.silver_trending_players')
    print(f"✅ Saved {len(df_trending)} trending players")
else:
    print("\n⚠️ No trending data to save")

print("\n" + "="*80)
print("✅ NEWS INGESTION COMPLETE")
print("="*80)

In [0]:
%sql
-- Quick validation of saved tables
SELECT 'bronze_player_news_raw' as table_name, COUNT(*) as row_count FROM main.fantasai.bronze_player_news_raw
UNION ALL
SELECT 'silver_player_news' as table_name, COUNT(*) as row_count FROM main.fantasai.silver_player_news
UNION ALL
SELECT 'silver_injury_reports' as table_name, COUNT(*) as row_count FROM main.fantasai.silver_injury_reports
UNION ALL
SELECT 'silver_trending_players' as table_name, COUNT(*) as row_count FROM main.fantasai.silver_trending_players

In [0]:
%sql
-- Preview recent player news
SELECT 
  player_name,
  position,
  team,
  injury_status,
  injury_notes,
  news_updated,
  depth_chart_order,
  depth_chart_position
FROM main.fantasai.silver_player_news
ORDER BY news_updated DESC
LIMIT 20

In [0]:
# Summary of player news ingestion
print("\n====================================================")
print("📰 Player News & Injury Ingestion Summary")
print("====================================================\n")

news_summary = spark.sql("""
SELECT 
  'Recent News' as data_type,
  COUNT(DISTINCT player_id) as unique_players,
  COUNT(*) as total_records,
  MAX(news_updated) as latest_update
FROM main.fantasai.silver_player_news
WHERE news_updated >= CURRENT_DATE() - INTERVAL 7 DAYS

UNION ALL

SELECT 
  'Active Injuries' as data_type,
  COUNT(DISTINCT player_id) as unique_players,
  COUNT(*) as total_records,
  MAX(fetched_at) as latest_update
FROM main.fantasai.silver_injury_reports
WHERE injury_status IS NOT NULL

UNION ALL

SELECT 
  'Trending Adds' as data_type,
  COUNT(DISTINCT player_id) as unique_players,
  COUNT(*) as total_records,
  MAX(fetched_at) as latest_update
FROM main.fantasai.silver_trending_players
""")

print("📊 Data Summary:")
display(news_summary)

injury_by_status = spark.sql("""
SELECT 
  injury_status,
  COUNT(*) as player_count,
  ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER (), 1) as pct_of_total
FROM main.fantasai.silver_injury_reports
GROUP BY injury_status
ORDER BY player_count DESC
""")

print("\n🏥 Injury Status Breakdown:")
display(injury_by_status)

print("\n✅ Player news ingestion complete")
print("\n📌 Data Sources:")
print("   - Sleeper API (free, no auth required)")
print("   - Player metadata, news, injuries, trending")
print("\n📅 Recommended Schedule:")
print("   - Run daily during season for injury updates")
print("   - Run multiple times per day on Sunday for game-day news")
print("\n💾 Output Tables:")
print("   - main.fantasai.bronze_player_news_raw (all players)")
print("   - main.fantasai.silver_player_news (recent news)")
print("   - main.fantasai.silver_injury_reports (injuries)")
print("   - main.fantasai.silver_trending_players (waiver adds)")